In [68]:
import torch
import numpy as np

@torch.no_grad()
def loo_node_importance_set_robust_gt(
    model,
    batch,
    pad_mask_true_means_pad=True,  # True => mask True means PAD/ignore
    positive_class_idx=1,          # keep as 1 for standard binary softmax
):
    """
    Returns:
      delta_gt: [B, N]   signed contribution to the GT class probability
                        + => node supports the correct class
                        - => node hurts the correct class (pushes toward the wrong class)
      p_true_base: [B]   base prob of the GT class
      p_pos_base:  [B]   base prob of positive class (class 1), for debugging/analysis
    """
    model.eval()
    device = getattr(model, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    x = batch["features"].to(device)  # [B,N,D]
    y = batch["labels"].to(device).long().view(-1)  # [B]
    pad_mask = None #batch.get("pad_mask", None)
    if pad_mask is not None:
        pad_mask = pad_mask.to(device).bool()

    B, N, D = x.shape

    # ----- base prediction -----
    logits_base = model(x, pad_mask if pad_mask is not None else None)   # [B,2]
    p_pos_base = torch.softmax(logits_base, dim=1)[:, positive_class_idx]  # P(y=1) [B]

    # prob of GT class: if y=1 -> p_pos; if y=0 -> 1-p_pos
    p_true_base = torch.where(y == 1, p_pos_base, 1.0 - p_pos_base)

    print(f"Base p_pos(mean)={p_pos_base.mean().item():.4f}, Base p_true(mean)={p_true_base.mean().item():.4f}, gt={y.detach().cpu().numpy()}")

    # ----- baseline for feature replacement (stronger than zero for standardized features) -----
    mean_vec = torch.zeros_like(x.mean(dim=1, keepdim=True)) #x.mean(dim=1, keepdim=True)  # [B,1,D]

    delta_gt = torch.zeros((B, N), device=device)

    for i in range(N):
        # replace node i features (robust even if mask is ignored)
        x_i = x.clone()
        x_i[:, i:i+1, :] = mean_vec

        # mask node i out (robust if mask is used)
        m_i = None
        if pad_mask is not None:
            m_i = pad_mask.clone()
            if pad_mask_true_means_pad:
                m_i[:, i] = True    # ignore node i
            else:
                m_i[:, i] = False   # ignore node i if True means VALID in your setup

        logits_i = model(x_i, m_i if pad_mask is not None else None)
        p_pos_i = torch.softmax(logits_i, dim=1)[:, positive_class_idx]
        p_true_i = torch.where(y == 1, p_pos_i, 1.0 - p_pos_i)

        # contribution to GT class prob (positive => supports correct label)
        delta_gt[:, i] = (p_true_base - p_true_i)

    return (
        delta_gt.detach().cpu().numpy(),
        p_true_base.detach().cpu().numpy(),
        p_pos_base.detach().cpu().numpy(),
    )


In [28]:
import sys
sys.path.append("..")
from models.MIL import RadiomicsMIL
from dataloaders.deep_dataloaders import get_dataloaders_deep_learning
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
import torch
import numpy as np
import os
from glob import glob
from pathlib import Path
import argparse 
from models.deep_sets import RadiomicsDeepSets
from models.transformer import RadiomicsTransformer
from models.MIL import RadiomicsMIL
from models.graph import RadiomicsGraph
from models.set_transformer import RadiomicsSetTransformer
from models.classical_ml import get_ml_models
from dataloaders.deep_dataloaders import get_dataloaders_deep_learning, get_center2_as_test_loader
from dataloaders.graph_dataloader import get_dataloaders_graph, get_center2_as_test_loader_graph
from dataloaders.ml_dataloaders import get_dataloaders_ml, get_classical_test_loader_center2
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
from pathlib import Path

def model_generator(model_name: str):
    if model_name == "transformer":
        return RadiomicsTransformer
    elif model_name == "deep_sets":
        return RadiomicsDeepSets
    elif model_name == "mil":
        return RadiomicsMIL
    elif model_name == "graph":
        return RadiomicsGraph
    elif model_name == "set_transformer":
        return RadiomicsSetTransformer
    else:
        raise ValueError(f"Model {model_name} not found in model zoo.")

def data_function_generator_dl(args):
    if args.model_name in ["transformer", "deep_sets", "mil", "set_transformer"]:
        return get_dataloaders_deep_learning, get_center2_as_test_loader, test_model
    elif args.model_name == "graph":
        return get_dataloaders_graph, get_center2_as_test_loader_graph, test_model_graph
    else:
        raise ValueError(f"Model {args.model_name} not found in data function generator.")


def get_trained_model(args, fold_index: int):
    config_base_dir = '../configs'
    model_configs = read_yaml_file(Path(config_base_dir) / f"{args.model_name}.yaml")
    args.batch_size = model_configs['batch_size']
    print(args.use_coords, args.use_demographic)
    # get dataloaders
    get_data_loaders, get_center2_loader, test_model_func = data_function_generator_dl(args)
    train_loader, val_loader, test_loader = get_dataloaders_deep_learning(args, fold_index=fold_index)
    center2_loader = get_center2_loader(args)
    # define input dimension
    sample_batch = next(iter(train_loader))
    if args.model_name == "graph":
        input_dim = sample_batch.num_node_features
    else:
        input_dim = sample_batch['features'].shape[-1]
    print(f"Input dimension: {input_dim}")
    MODEL = model_generator(args.model_name)
    model = MODEL.load_from_checkpoint(args.best_model_path, 
                            input_dim=input_dim,
                            config=model_configs)   
    return model, test_loader    

class ARGS:
    def __init__(self, model_name: str, use_coords: bool, use_demographic: bool, fold_index: int):
        self.data_root = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset"
        self.model_name = model_name
        self.use_coords = use_coords
        self.use_demographic = use_demographic
        self.batch_size = 32
        self.fold_index = fold_index
        model_postfic = "radiomics"
        if self.use_coords and not self.use_demographic:
            model_postfic = "coords_" + model_postfic
        elif self.use_demographic and not self.use_coords:
            model_postfic = "demographic_" + model_postfic
        elif self.use_coords and self.use_demographic:
            model_postfic = "coords_demographic_" + model_postfic
        
        self.best_model_path = f"/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/cleaned_code/Results/{self.model_name}_{model_postfic}/fold_{fold_index}/checkpoints/best.ckpt"





In [69]:
fold_index = 0
use_coords = True
use_demographic = True
model_name = "transformer"

args = ARGS(model_name=model_name, use_coords=use_coords, use_demographic=use_demographic, fold_index=fold_index)

model, test_loader = get_trained_model(args, fold_index=fold_index)
y_true, y_pred, y_prob = test_model(model, test_loader)
print(compute_classification_metrics(model_name, y_true, y_pred, y_prob))

True True
Train size: 102, Val size: 18, Test size: 31
Razavi Test size: 80
Input dimension: 112
{'model': 'transformer', 'accuracy': 0.8387096774193549, 'precision': 0.8, 'recall': 0.9411764705882353, 'specificity': np.float64(0.7142857142857143), 'f1_score': 0.8648648648648649, 'roc_auc': 0.8991596638655462}


/home/reza/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [70]:
iterator = iter(test_loader)

In [71]:
batch = next(iterator)

loo_node_importance_set_robust_gt(model, batch)

Base p_pos(mean)=0.3851, Base p_true(mean)=0.6149, gt=[0]


(array([[0.17348677, 0.19776928, 0.22561419, 0.23138595, 0.24021786]],
       dtype=float32),
 array([0.6149413], dtype=float32),
 array([0.3850587], dtype=float32))

In [72]:
batch = next(iter(test_loader))
pm = batch["pad_mask"]
print(pm.dtype, pm.min().item(), pm.max().item(), pm.float().mean().item())


torch.bool False False 0.0
